# M_CHIP2CHIP

In [1]:
%reset -f

In [2]:
from pynq import (PL,allocate, Overlay)
import numpy as np
from PIL import Image
from datetime import datetime

PL.reset()

ol = Overlay("master-zcu104.bit")

In [3]:
VDMA_S2MM = {
    "S2MM_VDMACR": 0x30,
    "S2MM_VDMASR": 0x34,
    "S2MM_VDMA_IRQ_MASK": 0x3C,
    "S2MM_REG_INDEX": 0x44,
    "S2MM_VSIZE": 0xA0,
    "S2MM_HSIZE": 0xA4,
    "S2MM_STRIDE": 0xA8,
    "S2MM_SA1": 0xAC,
    "S2MM_SA2": 0xB0,
    "S2MM_SA3": 0xB4,
    "S2MM_SA4": 0xB8,
    "S2MM_SA5": 0xBC,
    "S2MM_SA6": 0xC0,
    "S2MM_SA7": 0xC4,
    "S2MM_SA8": 0xC8,
    "S2MM_SA9": 0xCC,
    "S2MM_SA10": 0xD0,
    "S2MM_SA11": 0xD4,
    "S2MM_SA12": 0xD8,
    "S2MM_SA13": 0xDC,
    "S2MM_SA14": 0xE0,
    "S2MM_SA15": 0xE4,
    "S2MM_SA16": 0xE8,
}

In [4]:
def save_img(fname, tensor):
    img_tensor = np.squeeze(tensor,axis=2)  # Remove batch dim → [C, H, W]
    img = Image.fromarray(img_tensor.astype(np.uint8), mode='L')  # 'L' = 8-bit pixels, black and white
    save_path=f"{fname}.png"
    img.save(save_path)
    return save_path

def save_frame(sufix, frame):
    print(f"type(frame)={type(frame)},  \
          frame.shape={frame.shape},frame.dtype={frame.dtype}")
    unpacked_frame  = frame.view(dtype=np.uint8).reshape((480, 640, 1))
    print(f"type(unpacked_frame)={type(unpacked_frame)},\nunpacked_frame.shape={unpacked_frame.shape}, \
          \nunpacked_frame.dtype={unpacked_frame.dtype}")
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    #fname=f"{timestamp}-{sufix}"
    fname=f"image-output-{sufix}"
    save_img(fname=fname, tensor=unpacked_frame)

In [82]:
import time

VDMA_S2MM = {
    "S2MM_VDMACR": 0x30,
    "S2MM_VDMASR": 0x34,
    "S2MM_VDMA_IRQ_MASK": 0x3C,
    "S2MM_REG_INDEX": 0x44,
    "S2MM_VSIZE": 0xA0,
    "S2MM_HSIZE": 0xA4,
    "S2MM_STRIDE": 0xA8,
    "S2MM_SA1": 0xAC,
    "S2MM_SA2": 0xB0,
    "S2MM_SA3": 0xB4,
    "S2MM_SA4": 0xB8,
    "S2MM_SA5": 0xBC,
    "S2MM_SA6": 0xC0,
    "S2MM_SA7": 0xC4,
    "S2MM_SA8": 0xC8,
    "S2MM_SA9": 0xCC,
    "S2MM_SA10": 0xD0,
    "S2MM_SA11": 0xD4,
    "S2MM_SA12": 0xD8,
    "S2MM_SA13": 0xDC,
    "S2MM_SA14": 0xE0,
    "S2MM_SA15": 0xE4,
    "S2MM_SA16": 0xE8,
}

def vdma_readframes(_vdma, _stride, _v_size, _frame_rcv1, _frame_rcv2):
    # ----------------------------------------
    # Reset S2MM Channel
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_VDMACR"], 0x00000004)  # Reset
    time.sleep(0.01)
    _vdma.write(VDMA_S2MM["S2MM_VDMACR"], 0x00000001)  # Run/Stop = 1, circular mode = 0
    #_vdma.write(VDMA_S2MM["S2MM_VDMACR"], 0x00000003)  # Run/Stop = 1, circular mode = 1

    # ----------------------------------------
    # Set frame buffer base addresses
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_REG_INDEX"], 0x0)
    _vdma.write(VDMA_S2MM["S2MM_SA1"], _frame_rcv1.physical_address)
    _vdma.write(VDMA_S2MM["S2MM_SA2"], _frame_rcv2.physical_address)

    # ----------------------------------------
    # Set stride (bytes per row)
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_STRIDE"], _stride)

    # ----------------------------------------
    # Set horizontal size (in bytes)
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_HSIZE"], _stride)

    # ----------------------------------------
    # Set vertical size (in lines) to trigger transfer
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_VSIZE"], _v_size)

def dump_s2mm_status(_vdma):
    # Read relevant registers using map
    cr = _vdma.read(VDMA_S2MM["S2MM_VDMACR"])
    sr = _vdma.read(VDMA_S2MM["S2MM_VDMASR"])
    vsize = _vdma.read(VDMA_S2MM["S2MM_VSIZE"])
    hsize = _vdma.read(VDMA_S2MM["S2MM_HSIZE"])
    stride = _vdma.read(VDMA_S2MM["S2MM_STRIDE"])
    sa1 = _vdma.read(VDMA_S2MM["S2MM_SA1"])
    sa2 = _vdma.read(VDMA_S2MM["S2MM_SA2"])

    print("----- VDMA S2MM Status Dump -----")
    print(f"Control Reg     (0x{VDMA_S2MM['S2MM_VDMACR']:02X}): 0x{cr:08X}")
    print(f"Status Reg      (0x{VDMA_S2MM['S2MM_VDMASR']:02X}): 0x{sr:08X}")
    print(f"Vertical Size   (0x{VDMA_S2MM['S2MM_VSIZE']:02X}): {vsize}")
    print(f"Horizontal Size (0x{VDMA_S2MM['S2MM_HSIZE']:02X}): {hsize}")
    print(f"Stride          (0x{VDMA_S2MM['S2MM_STRIDE']:02X}): {stride}")
    print(f"S2MM_SA1        (0x{VDMA_S2MM['S2MM_SA1']:02X}): 0x{sa1:08X}")
    print(f"S2MM_SA2        (0x{VDMA_S2MM['S2MM_SA2']:02X}): 0x{sa2:08X}")

    # Decode common status bits (optional)
    status_bits = {
        0:  "HALTED",
        1:  "VDMA Internal Error",
        2:  "Slave Error",
        3:  "Decode Error",
        4:  "Start of Frame Early Error",
        5:  "End of Line Early Error",
        6:  "Start of Frame Late Error",
        10: "End of Line Late Error",
        12: "Frame Count Interrupt",
        13: "Delay Count Interrupt",
        14: "Error Interrupt",
        31: "DMA Internal Halted"
    }

    print("Status Flags:")
    for bit, description in status_bits.items():
        if sr & (1 << bit):
            print(f" - Bit {bit}: {description}")

    print("----------------------------------")

def  read_s2mm_reg(_vdma,reg):
    return _vdma.read(VDMA_S2MM[reg])

def  write_s2mm_reg(_vdma,reg,val):
    _vdma.write(VDMA_S2MM[reg], val)
    return _vdma.read(VDMA_S2MM[reg])

def wait_frame(_vdma,timeout_seconds = 2):
    start_time = time.time()
    while True:
        status=read_s2mm_reg(_vdma,'S2MM_VDMASR')
        if status & (1 << 12):
            break
        if time.time() - start_time > timeout_seconds:
            raise RuntimeError(f"No frame received in {timeout_seconds}: Bit 12 of status register is not set.")
        time.sleep(0.01)  # optional: small delay to reduce CPU usage
    #clear the flag
    write_s2mm_reg(_vdma,'S2MM_VDMASR',0x00001000)

In [6]:
import time

CONTROL_REG_MAP = {
    "CTRL": 0x00,
    "GIE":     0x04,
    "IER":     0x08,
    "ISR":     0x0C,
}
def dump_hls_ctrl(_hls):


    # Read relevant registers
    ctrl = _hls.read(CONTROL_REG_MAP["CTRL"])
    gie  = _hls.read(CONTROL_REG_MAP["GIE"])
    ier  = _hls.read(CONTROL_REG_MAP["IER"])
    isr  = _hls.read(CONTROL_REG_MAP["ISR"])

    print("----- hls Dump -----")
    print(f"Control signals                  0x{CONTROL_REG_MAP['CTRL']:02X}: 0x{ctrl:08X}")
    print(f"Global Interrupt Enable Register 0x{CONTROL_REG_MAP['GIE']:02X}: 0x{gie:08X}")
    print(f"IP Interrupt Enable Register     0x{CONTROL_REG_MAP['IER']:02X}: 0x{ier:08X}")
    print(f"IP Interrupt Status Register     0x{CONTROL_REG_MAP['ISR']:02X}: 0x{isr:08X}")

    # Decode common status bits (optional)
    status_bits = {
        0:  "ap_start (Read/Write/COH)",
        1:  "ap_done (Read/COR)",
        2:  "ap_idle (Read)",
        3:  "ap_ready (Read/COR)",
        7:  "auto_restart (Read/Write)",
        9:  "interrupt (Read)",
    }

    print("Control Flags:")
    for bit, description in status_bits.items():
        if ctrl & (1 << bit):
            print(f" - Bit {bit}: {description}")

    print("----------------------------------")

def write_hls_reg(_hls, reg,value):
    _hls.write(CONTROL_REG_MAP[reg],value)
    return _hls.read(CONTROL_REG_MAP[reg])

def read_hls_reg(_hls, reg):
    return _hls.read(CONTROL_REG_MAP[reg])

def wait_hls_done(_hls,timeout_seconds = 2):
    start_time = time.time()

    while True:
        status = read_hls_reg(_hls, 'CTRL')
        if (status & 1) or (status & 4):
            break
        if time.time() - start_time > timeout_seconds:
            raise TimeoutError("Timeout while waiting for HLS block to become idle or complete.")
        time.sleep(0.01)  # optional: small delay to reduce CPU usage
        

In [7]:
# ----------------------------------------
# VIVADO configuration
# ----------------------------------------
# VDMA
VDMA_BASE_ADDR = 0xA001_0000  # Replace with actual base address
VDMA_RANGE     = 0x0_8000    # 32K
#FRAME_BUFFERS  = 2  #buggy
FRAME_BUFFERS  = 1  
# ----------------------------------------
# FRAME configuration
# ----------------------------------------
WIDTH          = 160
HEIGHT         = 480
BPP            = 4           # Bytes per pixel (32bpp)
STRIDE         = WIDTH * BPP # Bytes per line
FRAME_SIZE     = STRIDE * HEIGHT

# ----------------------------------------
# IMG2AXIS
# ----------------------------------------
IMG2AXIS_BASE_ADDR = 0xA001_8000
IMG2AXIS_RANGE =0x0_8000

In [8]:
# ----------------------------------------
# Allocate destination buffer (for S2MM)
# ----------------------------------------
frame_rcv1 = allocate(shape=(HEIGHT, WIDTH), dtype=np.uint32)
frame_rcv2 = allocate(shape=(HEIGHT, WIDTH), dtype=np.uint32)
print(f"frame_rcv1= 0x{frame_rcv1.physical_address:08X}\n\
frame_rcv2= 0x{frame_rcv2.physical_address:08X}")

frame_rcv1= 0x60080000
frame_rcv2= 0x60280000


In [9]:
from pynq import MMIO
img2axis_mmio = MMIO(IMG2AXIS_BASE_ADDR, IMG2AXIS_RANGE)
# ----------------------------------------
vdma_mmio = MMIO(VDMA_BASE_ADDR, VDMA_RANGE)

In [10]:
dump_hls_ctrl(img2axis_mmio)

----- hls Dump -----
Control signals                  0x00: 0x0000000E
Global Interrupt Enable Register 0x04: 0x00000000
IP Interrupt Enable Register     0x08: 0x00000000
IP Interrupt Status Register     0x0C: 0x00000000
Control Flags:
 - Bit 1: ap_done (Read/COR)
 - Bit 2: ap_idle (Read)
 - Bit 3: ap_ready (Read/COR)
----------------------------------


In [11]:
dump_s2mm_status(vdma_mmio)

----- VDMA S2MM Status Dump -----
Control Reg     (0x30): 0x00010001
Status Reg      (0x34): 0x00011000
Vertical Size   (0xA0): 480
Horizontal Size (0xA4): 640
Stride          (0xA8): 640
S2MM_SA1        (0xAC): 0x60080000
S2MM_SA2        (0xB0): 0x60280000
Status Flags:
 - Bit 12: Frame Count Interrupt
----------------------------------


In [92]:
#start vdma
#vdma_readframes(vdma_mmio,STRIDE,FRAME_BUFFERS*HEIGHT,frame_rcv1,frame_rcv2)


In [103]:
frame_rcv1.fill(0)
#start vdma
vdma_readframes(vdma_mmio,STRIDE,FRAME_BUFFERS*HEIGHT,frame_rcv1,frame_rcv2)
write_hls_reg(img2axis_mmio,'CTRL',1) #start hls
#dump_hls_ctrl(img2axis_mmio)
wait_hls_done(img2axis_mmio)
#dump_s2mm_status(vdma_mmio)
wait_frame(vdma_mmio)
save_frame('recv1',frame_rcv1)

type(frame)=<class 'pynq.buffer.PynqBuffer'>,            frame.shape=(480, 160),frame.dtype=uint32
type(unpacked_frame)=<class 'pynq.buffer.PynqBuffer'>,
unpacked_frame.shape=(480, 640, 1),           
unpacked_frame.dtype=uint8


-----

In [100]:
dump_s2mm_status(vdma_mmio)
wait_frame(vdma_mmio)
save_frame('recv1',frame_rcv1)


----- VDMA S2MM Status Dump -----
Control Reg     (0x30): 0x00010001
Status Reg      (0x34): 0x00011000
Vertical Size   (0xA0): 480
Horizontal Size (0xA4): 640
Stride          (0xA8): 640
S2MM_SA1        (0xAC): 0x60080000
S2MM_SA2        (0xB0): 0x60280000
Status Flags:
 - Bit 12: Frame Count Interrupt
----------------------------------
type(frame)=<class 'pynq.buffer.PynqBuffer'>,            frame.shape=(480, 160),frame.dtype=uint32
type(unpacked_frame)=<class 'pynq.buffer.PynqBuffer'>,
unpacked_frame.shape=(480, 640, 1),           
unpacked_frame.dtype=uint8


In [80]:
f"{write_s2mm_reg(vdma_mmio,'S2MM_VDMASR',0x00001000):08x}"

'00010000'

In [81]:
dump_s2mm_status(vdma_mmio)

----- VDMA S2MM Status Dump -----
Control Reg     (0x30): 0x00010001
Status Reg      (0x34): 0x00010000
Vertical Size   (0xA0): 480
Horizontal Size (0xA4): 640
Stride          (0xA8): 640
S2MM_SA1        (0xAC): 0x60080000
S2MM_SA2        (0xB0): 0x60280000
Status Flags:
----------------------------------
